# 21 — Perbandingan generasi YOLO (nano) pada protokol detektor terkunci

Arm `real_only` (301 foto train, DSC_0093 dikeluarkan; 87 validation). Tujuh generasi skala
nano/tiny: yolov8n, yolov5nu, yolov9t, yolov10n, yolo11n, yolo12n, yolo26n. Trainer `train_detector.py` (sha256 5b273050…) tidak diubah:
3000 update, batch 8, AdamW lr 1e-4, cosine, flip deterministik, letterbox 640, validasi tiap
250 update, checkpoint terbaik menurut mAP50:95 validation. Hanya jaringan (config + bobot COCO)
yang berbeda. Seed [0, 1, 2, 3, 4]. **Hanya validation; test tidak dibaca.**

Citra train di-letterbox sekali ke PNG 640 (diverifikasi identik piksel dan kotak dengan jalur
asli) agar pemuatan data tidak menjadi leher botol. 2 proses per GPU.
Batas waktu peluncuran job baru: 240 menit sejak sel scheduler mulai.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess, hashlib, time
WORK = Path('/kaggle/working/yolo_versions_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
archives = list(INPUT.rglob('texture_study_code.zip'))
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
TRAINER = WORK/'code/14_Benchmark_Kaggle/tools/train_detector.py'
def unique(marker):
    hits = sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits) != 1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
digest = hashlib.sha256((DATASET/'manifest.csv').read_bytes()).hexdigest()
if digest != '185e14221e8882f7bcabb5fbb455d6a64aa09378f07234e1ccd4f855531df6ba':
    raise ValueError(f'dataset manifest {digest} is not regrouped v4')
print('dataset', DATASET, digest[:16])

In [ ]:
import importlib.util, importlib.metadata as meta
if importlib.util.find_spec('ultralytics') is None or meta.version('ultralytics') != '8.4.163':
    subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.163'],check=True)
import torch
for package in ('torch','torchvision','ultralytics'): print(package, meta.version(package))
if torch.cuda.device_count() < 1: raise RuntimeError('GPU required')
N_GPU = torch.cuda.device_count()
print('GPUs', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

In [ ]:
HELPERS = {
"train_detector_versions.py": "\"\"\"Run the locked detector trainer with another YOLO generation.\n\n    python train_detector_versions.py --trainer <14_Benchmark_Kaggle/tools/train_detector.py> \\\n        --arm <arm> --seed 0 --updates 3000 --eval-every 250 --architecture yolo11n --out <out>\n\n`train_detector.py` (sha256 5b273050...) is the trainer behind every published detector number\nand is not edited. This wrapper only adds entries to its `ARCHITECTURES` table and then calls its\n`main`, so the loop, batches, flip rule, AdamW, cosine schedule, letterbox, update budget,\nperiodic validation and checkpoint rule are byte-for-byte the same for every generation.\nOnly the network (config + COCO-pretrained weights) differs. The test split is never read.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport sys\nfrom pathlib import Path\n\n# Nano / tiny scale of each generation shipped with Ultralytics 8.4. The 'u' variant of YOLOv5\n# is the anchor-free Ultralytics port; the original anchor-based YOLOv5 is not trainable here.\nVERSIONS = {\n    'yolov5nu': dict(cfg='yolov5nu.yaml', weights='yolov5nu.pt'),\n    'yolov8n': dict(cfg='yolov8n.yaml', weights='yolov8n.pt'),\n    'yolov9t': dict(cfg='yolov9t.yaml', weights='yolov9t.pt'),\n    'yolov10n': dict(cfg='yolov10n.yaml', weights='yolov10n.pt'),\n    'yolo11n': dict(cfg='yolo11n.yaml', weights='yolo11n.pt'),\n    'yolo12n': dict(cfg='yolo12n.yaml', weights='yolo12n.pt'),\n    'yolo26n': dict(cfg='yolo26n.yaml', weights='yolo26n.pt'),\n}\nLOCKED_TRAINER_SHA256 = '5b27305000a529a0b0c375be26aee83aec3ffb408ca9f0ab811d2f1329b5d61b'\n\n\ndef load_trainer(path):\n    path = Path(path).resolve()\n    digest = hashlib.sha256(path.read_bytes()).hexdigest()\n    if digest != LOCKED_TRAINER_SHA256:\n        raise SystemExit(f'{path} is not the locked trainer (sha256 {digest})')\n    spec = importlib.util.spec_from_file_location('train_detector', path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules['train_detector'] = module\n    spec.loader.exec_module(module)\n    for name, entry in VERSIONS.items():\n        module.ARCHITECTURES.setdefault(name, entry)\n    return module\n\n\ndef main(argv=None):\n    argv = list(sys.argv[1:] if argv is None else argv)\n    if '--trainer' not in argv:\n        raise SystemExit('--trainer is required')\n    i = argv.index('--trainer')\n    trainer = load_trainer(argv[i + 1])\n    del argv[i:i + 2]\n    # `choices` of --architecture was fixed when argparse was built inside main(); rebuild it\n    # by calling main after the table is extended (choices=sorted(ARCHITECTURES) is evaluated\n    # at call time, so the new names are accepted).\n    trainer.main(argv)\n\n\nif __name__ == '__main__':\n    main()\n",
"preletterbox_arm.py": "\"\"\"Write a copy of an arm whose TRAIN images are already letterboxed to 640 (lossless PNG).\n\n    python preletterbox_arm.py --trainer <train_detector.py> --arm <arm> --out <new arm> --verify\n\nThe locked trainer decodes every 2048x1366 JPEG and letterboxes it again at every update,\nwhich makes a 3000-update run CPU-bound. Letterboxing a 640x640 canvas a second time is the\nidentity in the trainer (PIL returns a copy when the size is unchanged, and the offsets are 0),\nso the tensors the network sees are unchanged. `--verify` checks this for every training image,\nwith and without the horizontal flip: pixels must be equal and box coordinates equal within\n1e-9. Validation images are copied untouched, because Ultralytics validation reads them with\nits own loader. The test split is not read.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom train_detector_versions import load_trainer  # noqa: E402\n\n\ndef build(trainer, arm, out):\n    arm, out = Path(arm), Path(out)\n    if out.exists():\n        raise SystemExit(f'{out} exists; use a new directory')\n    for split in ('train', 'val'):\n        (out / 'images' / split).mkdir(parents=True)\n        (out / 'labels' / split).mkdir(parents=True)\n    source = trainer.ArmDataset(arm, 'train')\n    for index in range(len(source)):\n        stem, image, boxes = source.load(index, flip=False)\n        image.save(out / 'images/train' / f'{stem}.png')\n        (out / 'labels/train' / f'{stem}.txt').write_text(\n            ''.join(f'{c} {x!r} {y!r} {w!r} {h!r}\\n' for c, x, y, w, h in boxes))\n    for path in sorted((arm / 'images/val').glob('*')):\n        shutil.copyfile(path, out / 'images/val' / path.name)\n    for path in sorted((arm / 'labels/val').glob('*')):\n        shutil.copyfile(path, out / 'labels/val' / path.name)\n    manifest = json.loads((arm / 'arm_manifest.json').read_text())\n    manifest['preletterboxed_from'] = str(arm.resolve())\n    (out / 'arm_manifest.json').write_text(json.dumps(manifest, indent=2) + '\\n')\n    (out / 'data.yaml').write_text((arm / 'data.yaml').read_text().replace(\n        f'path: {arm.resolve()}', f'path: {out.resolve()}'))\n    return len(source)\n\n\ndef verify(trainer, arm, out):\n    a, b = trainer.ArmDataset(Path(arm), 'train'), trainer.ArmDataset(Path(out), 'train')\n    if [p.stem for p in a.items] != [p.stem for p in b.items]:\n        raise SystemExit('train item order differs; batch indices would not match')\n    worst = 0.0\n    for index in range(len(a)):\n        for flip in (False, True):\n            sa, ia, ba = a.load(index, flip)\n            sb, ib, bb = b.load(index, flip)\n            if sa != sb or not np.array_equal(np.asarray(ia), np.asarray(ib)):\n                raise SystemExit(f'pixels differ for {sa} flip={flip}')\n            if len(ba) != len(bb) or any(x[0] != y[0] for x, y in zip(ba, bb)):\n                raise SystemExit(f'boxes differ for {sa}')\n            worst = max([worst] + [abs(u - v) for x, y in zip(ba, bb) for u, v in zip(x[1:], y[1:])])\n    if worst > 1e-9:\n        raise SystemExit(f'box coordinates differ by {worst}')\n    return dict(images=len(a), max_box_difference=worst)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--trainer', required=True)\n    parser.add_argument('--arm', required=True)\n    parser.add_argument('--out', required=True)\n    parser.add_argument('--verify', action='store_true')\n    args = parser.parse_args(argv)\n    trainer = load_trainer(args.trainer)\n    print('written', build(trainer, args.arm, args.out))\n    if args.verify:\n        print('verified', verify(trainer, args.arm, args.out))\n\n\nif __name__ == '__main__':\n    main()\n"
}
DIGESTS = {"train_detector_versions.py": "de127d998c42fbce4114f028f0f116fa5c2e7af77f1775ed2784bc54d9f9c949", "preletterbox_arm.py": "371a9c31f535cdc2b9dec71b5b0fe96c74f304e3980f0763a98fa3a2f255aadb"}
TOOLS = WORK/'yolo_tools'; TOOLS.mkdir()
for name, text in HELPERS.items():
    (TOOLS/name).write_text(text)
    assert hashlib.sha256(text.encode()).hexdigest() == DIGESTS[name]
    print(name, DIGESTS[name][:16])

In [ ]:
# real_only arm (same builder and exclusion as balancing experiment C), then the 640 copy.
ENV = {**os.environ, 'PYTHONPATH': str(HERE), 'MELON_DATASET': str(DATASET), 'PYTHONUNBUFFERED': '1'}
subprocess.run([sys.executable, str(HERE/'tools/build_balance_arms.py'), '--real-only', '--out', str(WORK/'arms')],
               cwd=HERE, env=ENV, check=True)
subprocess.run([sys.executable, str(TOOLS/'preletterbox_arm.py'), '--trainer', str(TRAINER),
                '--arm', str(WORK/'arms/real_only'), '--out', str(WORK/'arms640/real_only'), '--verify'],
               cwd=TOOLS, env=ENV, check=True)
arm_manifest = json.loads((WORK/'arms640/real_only/arm_manifest.json').read_text())
print(arm_manifest['counts'])
assert arm_manifest['counts']['total_per_class'] == {'0': 217, '1': 84} and arm_manifest['counts']['validation'] == 87

In [ ]:
VERSIONS = ['yolov8n', 'yolov5nu', 'yolov9t', 'yolov10n', 'yolo11n', 'yolo12n', 'yolo26n']
SEEDS = [0, 1, 2, 3, 4]
WORKERS_PER_GPU = 2
BUDGET_MINUTES = 240

# COCO weights fetched once; every job runs with this folder as working directory.
WEIGHTS = WORK/'weights'; WEIGHTS.mkdir()
from ultralytics import YOLO
for name in VERSIONS:
    spec = {'yolov5nu':'yolov5nu.pt','yolov8n':'yolov8n.pt','yolov9t':'yolov9t.pt','yolov10n':'yolov10n.pt',
            'yolo11n':'yolo11n.pt','yolo12n':'yolo12n.pt','yolo26n':'yolo26n.pt'}[name]
    cwd = os.getcwd(); os.chdir(WEIGHTS)
    try: YOLO(spec)
    finally: os.chdir(cwd)
print(sorted(p.name for p in WEIGHTS.glob('*.pt')))
# One private arm copy per worker slot, so concurrent validations never share a label cache.
SLOTS = [(g, w) for g in range(N_GPU) for w in range(WORKERS_PER_GPU)]
for g, w in SLOTS:
    dst = WORK/f'arm_slot{g}{w}/real_only'
    shutil.copytree(WORK/'arms640/real_only', dst)
    (dst/'data.yaml').write_text((dst/'data.yaml').read_text().replace(
        str((WORK/'arms640/real_only').resolve()), str(dst.resolve())))
print('slots', SLOTS)

In [ ]:
# Seed-major order: if the budget stops new launches, every finished seed covers all generations.
import collections
jobs = collections.deque((name, seed) for seed in SEEDS for name in VERSIONS)
RUNS = WORK/'runs'; LOGS = WORK/'logs'; RUNS.mkdir(); LOGS.mkdir()
running, finished, failed, skipped = {}, [], [], []
started = time.time()
def launch(slot, name, seed):
    g, w = slot
    argv = [sys.executable, str(TOOLS/'train_detector_versions.py'), '--trainer', str(TRAINER),
            '--arm', str(WORK/f'arm_slot{g}{w}/real_only'), '--seed', str(seed), '--updates', '3000',
            '--eval-every', '250', '--architecture', name, '--out', str(RUNS/name)]
    env = {**ENV, 'CUDA_VISIBLE_DEVICES': str(g), 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
    log = open(LOGS/f'{name}__seed{seed}.log', 'w')
    proc = subprocess.Popen(argv, cwd=WEIGHTS, env=env, stdout=log, stderr=subprocess.STDOUT)
    return dict(proc=proc, name=name, seed=seed, t0=time.time(), log=log)
free = list(SLOTS)
while jobs or running:
    while free and jobs:
        if (time.time() - started) / 60 > BUDGET_MINUTES:
            skipped.extend(jobs); jobs.clear(); break
        slot = free.pop(0); name, seed = jobs.popleft()
        running[slot] = launch(slot, name, seed)
    time.sleep(10)
    for slot, job in list(running.items()):
        code = job['proc'].poll()
        if code is None: continue
        job['log'].close(); minutes = round((time.time() - job['t0']) / 60, 1)
        (finished if code == 0 else failed).append(dict(name=job['name'], seed=job['seed'], gpu=slot[0],
                                                        code=code, minutes=minutes))
        print(time.strftime('%H:%M:%S'), job['name'], job['seed'], 'code', code, minutes, 'min', flush=True)
        del running[slot]; free.append(slot)
wall = round((time.time() - started) / 60, 1)
print('wall minutes', wall, 'finished', len(finished), 'failed', failed, 'skipped', list(skipped))
for f in failed: print((LOGS/f"{f['name']}__seed{f['seed']}.log").read_text()[-2000:])

In [ ]:
# Summary per generation (selected checkpoint and final state) and paired differences vs YOLOv8n.
import statistics
def read(name, seed):
    path = RUNS/name/f'real_only__seed{seed}'/'run_summary.json'
    return json.loads(path.read_text()) if path.exists() else None
table = {}
for name in VERSIONS:
    runs = {s: read(name, s) for s in SEEDS}
    runs = {s: r for s, r in runs.items() if r}
    if not runs: continue
    sel = {s: r['selected']['validation'] for s, r in runs.items()}
    fin = {s: r['validation'] for s, r in runs.items()}
    def stats(d, key):
        values = [key(v) for v in d.values()]
        return dict(mean=statistics.fmean(values), sd=statistics.pstdev(values), per_seed=dict(zip(d, values)))
    table[name] = dict(seeds=sorted(runs),
        selected=dict(map50_95=stats(sel, lambda v: v['map50_95']), map50=stats(sel, lambda v: v['map50']),
                      ap_unripe=stats(sel, lambda v: v['per_class']['unripe']['ap50_95']),
                      recall_unripe=stats(sel, lambda v: v['recall'][1]),
                      update=dict(per_seed={s: r['selected']['update'] for s, r in runs.items()})),
        final=dict(map50_95=stats(fin, lambda v: v['map50_95']),
                   ap_unripe=stats(fin, lambda v: v['per_class']['unripe']['ap50_95'])),
        training_seconds=stats(runs, lambda r: r['training_seconds']))
ref = table.get('yolov8n')
for name, row in table.items():
    if not ref or name == 'yolov8n': continue
    common = [s for s in row['seeds'] if s in ref['seeds']]
    diffs = [row['selected']['map50_95']['per_seed'][s] - ref['selected']['map50_95']['per_seed'][s] for s in common]
    row['paired_vs_yolov8n'] = dict(seeds=common, mean_diff=statistics.fmean(diffs) if diffs else None,
                                    wins=sum(d > 0 for d in diffs), diffs=diffs)
OUT = WORK/'result'; OUT.mkdir()
report = dict(versions=VERSIONS, seeds=SEEDS, arm='real_only', updates=3000, eval_every=250, batch=8, lr=1e-4,
              optimizer='AdamW (identical for all; not each generation\'s default)', ultralytics=meta.version('ultralytics'),
              workers_per_gpu=WORKERS_PER_GPU, gpus=[torch.cuda.get_device_name(i) for i in range(N_GPU)],
              finished=finished, failed=failed, skipped=list(skipped), wall_minutes=wall, table=table,
              helper_sha256=DIGESTS, split='valid', test_used=False)
(OUT/'results.json').write_text(json.dumps(report, indent=2) + '\n')
for name, row in table.items():
    s = row['selected']
    print(f"{name:9s} n={len(row['seeds'])} mAP50:95 {s['map50_95']['mean']:.4f}±{s['map50_95']['sd']:.4f} "
          f"AP_unripe {s['ap_unripe']['mean']:.4f} R_unripe {s['recall_unripe']['mean']:.3f} "
          f"train {row['training_seconds']['mean']/60:.1f} min", row.get('paired_vs_yolov8n', {}).get('wins'))
# Keep run folders (summaries, histories, predictions, weights); drop bulky arm copies and code.
shutil.copytree(RUNS, OUT/'runs'); shutil.copytree(LOGS, OUT/'logs')
for p in [WORK/'arms', WORK/'arms640', WORK/'code', WORK/'weights', RUNS, LOGS, *WORK.glob('arm_slot*')]:
    shutil.rmtree(p, ignore_errors=True)
print(sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 1e6, 'MB in result')